# 🏷️ YOLO 路牌辨識訓練
## 台北科技大學 多媒體應用 Project 6

---
**資料集**: `_SignDetection.yolo26` — 151 張圖片

**類別 (4類)**:
| ID | 類別名稱 | 說明 |
|:--:|:--------:|:----:|
| 0 | `blocked` | 道路封閉 |
| 1 | `pedestrian` | 行人號誌 |
| 2 | `rail` | 鐵路平交道 |
| 3 | `stop` | 停車再開 |

---
> ⚠️ **提示**: 若在 Google Colab 執行，請先將 `_SignDetection.yolo26` 資料夾上傳至 `/content/`

## 📦 Step 1 — 安裝 ultralytics

In [ ]:
# 安裝 ultralytics (包含 YOLOv8/v11)
!pip install ultralytics -q

import ultralytics
ultralytics.checks()
print(f'ultralytics 版本: {ultralytics.__version__}')

## 📁 Step 2 — 設定路徑

In [ ]:
import os
import yaml
from pathlib import Path

# ─── 自動偵測執行環境 ────────────────────────────────────────
IN_COLAB = 'google.colab' in str(get_ipython())
print(f'執行環境: {"Google Colab" if IN_COLAB else "本地 Jupyter"}')

# ─── 路徑設定 ────────────────────────────────────────────────
if IN_COLAB:
    # Colab: 從 Google Drive 或 /content 讀取
    PROJECT_ROOT = Path('/content')
else:
    # 本地: 與 notebook 同一目錄
    PROJECT_ROOT = Path('.').resolve()

DATASET_DIR = PROJECT_ROOT / '_SignDetection.yolo26'
DATA_YAML   = DATASET_DIR  / 'data.yaml'
RUNS_DIR    = PROJECT_ROOT / 'runs' / 'sign_detection'

print(f'Project 根目錄: {PROJECT_ROOT}')
print(f'資料集路徑    : {DATASET_DIR}')
print(f'Data YAML    : {DATA_YAML}')
print(f'資料集存在   : {DATASET_DIR.exists()}')

## 🔧 Step 3 — 修復 data.yaml 路徑

In [ ]:
# 原始 data.yaml 使用相對路徑，這裡改為絕對路徑避免問題

with open(DATA_YAML, 'r', encoding='utf-8') as f:
    cfg = yaml.safe_load(f)

train_img_dir = DATASET_DIR / 'train' / 'images'
valid_img_dir = DATASET_DIR / 'valid' / 'images'
test_img_dir  = DATASET_DIR / 'test'  / 'images'

cfg['train'] = str(train_img_dir)
cfg['val']   = str(valid_img_dir) if valid_img_dir.exists() else str(train_img_dir)
cfg['test']  = str(test_img_dir)  if test_img_dir.exists()  else str(train_img_dir)

FIXED_YAML = DATASET_DIR / 'data_fixed.yaml'
with open(FIXED_YAML, 'w', encoding='utf-8') as f:
    yaml.dump(cfg, f, allow_unicode=True, sort_keys=False)

print('data.yaml 內容 (修復後):')  
print('─' * 40)
for k, v in cfg.items():
    print(f'  {k:8s}: {v}')
print('─' * 40)

## 📊 Step 4 — 資料集統計

In [ ]:
import collections
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import cv2, random
import numpy as np

CLASS_NAMES = ['blocked', 'pedestrian', 'rail', 'stop']
CLASS_COLORS = ['#FF6B6B', '#4ECDC4', '#FFE66D', '#A8E6CF']

# ─── 統計各類別標籤數量 ──────────────────────────────────────
label_dir = DATASET_DIR / 'train' / 'labels'
class_count = collections.Counter()

for lbl_file in label_dir.glob('*.txt'):
    with open(lbl_file, 'r') as f:
        for line in f:
            cls_id = int(line.strip().split()[0])
            class_count[cls_id] += 1

print('='*50)
print('  📦 資料集統計')
print('='*50)
print(f'  訓練圖片數: {len(list((DATASET_DIR/"train"/"images").glob("*.jpg")))}')
print(f'  標籤檔數  : {len(list(label_dir.glob("*.txt")))}')
print()
print('  各類別標注數量:')
for cls_id, name in enumerate(CLASS_NAMES):
    count = class_count.get(cls_id, 0)
    bar = '█' * (count // 5)
    print(f'    {cls_id} {name:12s}: {count:4d}  {bar}')
print('='*50)

# ─── 繪製圓餅圖 ──────────────────────────────────────────────
counts = [class_count.get(i, 0) for i in range(len(CLASS_NAMES))]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.patch.set_facecolor('#1a1a2e')

# 圓餅圖
ax = axes[0]
ax.set_facecolor('#1a1a2e')
wedges, texts, autotexts = ax.pie(
    counts, labels=CLASS_NAMES, autopct='%1.1f%%',
    colors=CLASS_COLORS, startangle=90,
    wedgeprops=dict(edgecolor='#1a1a2e', linewidth=2),
    textprops=dict(color='white', fontsize=11)
)
for t in autotexts: t.set_color('#1a1a2e')
ax.set_title('類別分佈', color='white', fontsize=13, pad=15)

# 長條圖
ax2 = axes[1]
ax2.set_facecolor('#16213e')
bars = ax2.bar(CLASS_NAMES, counts, color=CLASS_COLORS, edgecolor='#1a1a2e', linewidth=1.5)
for bar, count in zip(bars, counts):
    ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
             str(count), ha='center', va='bottom', color='white', fontsize=11, fontweight='bold')
ax2.set_ylabel('標注數量', color='white', fontsize=11)
ax2.set_title('各類別標注數量', color='white', fontsize=13)
ax2.tick_params(colors='white')
ax2.spines['bottom'].set_color('#444')
ax2.spines['left'].set_color('#444')
ax2.spines['top'].set_visible(False)
ax2.spines['right'].set_visible(False)
ax2.set_facecolor('#16213e')

plt.tight_layout()
plt.savefig(str(PROJECT_ROOT / 'dataset_stats.png'), dpi=120, bbox_inches='tight')
plt.show()
print('統計圖已儲存 → dataset_stats.png')

## 🖼️ Step 5 — 預覽訓練圖片（含標注框）

In [ ]:
def draw_yolo_label(img_path, label_path, class_names, colors):
    """在圖片上繪製 YOLO 格式的標注框"""
    img = cv2.imread(str(img_path))
    if img is None:
        return None
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    h, w = img.shape[:2]

    if label_path.exists():
        with open(label_path, 'r') as f:
            for line in f:
                parts = line.strip().split()
                if len(parts) < 5: continue
                cls_id = int(parts[0])
                cx, cy, bw, bh = map(float, parts[1:5])
                # 轉換為像素座標
                x1 = int((cx - bw/2) * w)
                y1 = int((cy - bh/2) * h)
                x2 = int((cx + bw/2) * w)
                y2 = int((cy + bh/2) * h)
                color = [int(c * 255) for c in plt.cm.get_cmap('Set1')(cls_id)[:3]]
                cv2.rectangle(img, (x1, y1), (x2, y2), color, 2)
                label = class_names[cls_id] if cls_id < len(class_names) else str(cls_id)
                cv2.putText(img, label, (x1, y1-5), cv2.FONT_HERSHEY_SIMPLEX, 0.6, color, 2)
    return img

# 隨機選取 9 張圖片預覽
img_dir = DATASET_DIR / 'train' / 'images'
lbl_dir = DATASET_DIR / 'train' / 'labels'
img_files = list(img_dir.glob('*.jpg'))
selected = random.sample(img_files, min(9, len(img_files)))

fig, axes = plt.subplots(3, 3, figsize=(15, 12))
fig.patch.set_facecolor('#1a1a2e')
fig.suptitle('訓練資料集預覽 (含標注框)', color='white', fontsize=16, fontweight='bold', y=1.01)

for ax, img_path in zip(axes.flatten(), selected):
    lbl_path = lbl_dir / (img_path.stem + '.txt')
    img = draw_yolo_label(img_path, lbl_path, CLASS_NAMES, CLASS_COLORS)
    if img is not None:
        ax.imshow(img)
    ax.set_facecolor('#1a1a2e')
    ax.axis('off')
    ax.set_title(img_path.stem[:20], color='#aaa', fontsize=7)

plt.tight_layout()
plt.savefig(str(PROJECT_ROOT / 'dataset_preview.png'), dpi=120, bbox_inches='tight')
plt.show()
print('預覽圖已儲存 → dataset_preview.png')

## 🚀 Step 6 — 訓練模型

In [ ]:
from ultralytics import YOLO

# ─── 訓練設定 ────────────────────────────────────────────────
# 模型選擇說明:
#   yolo11n.pt  → 最小最快，適合 JetBot (推薦)
#   yolo11s.pt  → 小型，精度較高
#   yolo11m.pt  → 中型，需要較多記憶體
MODEL_NAME = 'yolo11n.pt'

EPOCHS  = 100   # 訓練輪數
IMGSZ   = 416   # 圖片大小 (與 JetBot TRT_YOLO 一致)
BATCH   = 16    # Batch size (若 OOM 請改小, 如 8)
WORKERS = 4     # DataLoader 工作執行緒數

# ─── 載入預訓練模型並開始訓練 ────────────────────────────────
model = YOLO(MODEL_NAME)

results = model.train(
    # ── 資料 ──
    data    = str(FIXED_YAML),
    imgsz   = IMGSZ,
    batch   = BATCH,
    workers = WORKERS,
    
    # ── 訓練 ──
    epochs  = EPOCHS,
    patience= 30,             # early stopping: 30 epoch 無改善則停止
    optimizer='SGD',
    lr0     = 0.01,
    lrf     = 0.01,
    momentum= 0.937,
    weight_decay = 0.0005,
    warmup_epochs= 3.0,
    
    # ── 損失 ──
    box = 7.5,
    cls = 0.5,
    dfl = 1.5,
    
    # ── 資料增強 ──
    hsv_h   = 0.015,
    hsv_s   = 0.7,
    hsv_v   = 0.4,
    fliplr  = 0.5,
    mosaic  = 1.0,
    scale   = 0.5,
    translate = 0.1,
    
    # ── 輸出 ──
    project     = str(PROJECT_ROOT / 'runs'),
    name        = 'sign_detection',
    exist_ok    = True,
    save        = True,
    save_period = 10,         # 每 10 epoch 存一次
    plots       = True,
    verbose     = True,
    device      = '',         # 自動選擇 GPU/CPU
)

print('\n訓練完成!')
print(f'最佳模型: {PROJECT_ROOT}/runs/sign_detection/weights/best.pt')

## 📊 Step 7 — 驗證模型

In [ ]:
# 載入最佳模型進行驗證
best_model_path = PROJECT_ROOT / 'runs' / 'sign_detection' / 'weights' / 'best.pt'

model_val = YOLO(str(best_model_path))

metrics = model_val.val(
    data    = str(FIXED_YAML),
    imgsz   = IMGSZ,
    conf    = 0.25,
    iou     = 0.6,
    plots   = True,
    verbose = True,
)

print('\n' + '='*50)
print('  📊 驗證結果')
print('='*50)
print(f'  mAP50    : {metrics.box.map50:.4f}  ({metrics.box.map50*100:.1f}%)')
print(f'  mAP50-95 : {metrics.box.map:.4f}  ({metrics.box.map*100:.1f}%)')
print(f'  Precision: {metrics.box.mp:.4f}')
print(f'  Recall   : {metrics.box.mr:.4f}')
print('='*50)

## 🔍 Step 8 — 推論測試

In [ ]:
# 對訓練圖片做推論測試 (取前 6 張)
img_files = sorted((DATASET_DIR / 'train' / 'images').glob('*.jpg'))[:6]

results = model_val.predict(
    source  = [str(p) for p in img_files],
    imgsz   = IMGSZ,
    conf    = 0.25,
    iou     = 0.45,
    verbose = False,
)

# 視覺化結果
fig, axes = plt.subplots(2, 3, figsize=(15, 10))
fig.patch.set_facecolor('#1a1a2e')
fig.suptitle('推論結果', color='white', fontsize=16, fontweight='bold')

COLOR_MAP = {
    0: (255, 100, 100),   # blocked   → 紅
    1: (100, 220, 200),   # pedestrian → 青
    2: (255, 230, 100),   # rail      → 黃
    3: (160, 230, 180),   # stop      → 綠
}

for ax, result in zip(axes.flatten(), results):
    img = cv2.cvtColor(cv2.imread(str(result.path)), cv2.COLOR_BGR2RGB)
    h, w = img.shape[:2]

    detections = []
    if result.boxes is not None and len(result.boxes) > 0:
        for box in result.boxes:
            cls_id = int(box.cls[0])
            conf_v = float(box.conf[0])
            xyxy   = [int(v) for v in box.xyxy[0]]
            color  = COLOR_MAP.get(cls_id, (255, 255, 255))
            cv2.rectangle(img, (xyxy[0], xyxy[1]), (xyxy[2], xyxy[3]), color, 3)
            label = f"{CLASS_NAMES[cls_id]} {conf_v:.2f}"
            cv2.putText(img, label, (xyxy[0], xyxy[1]-8),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.65, color, 2)
            detections.append(f"{CLASS_NAMES[cls_id]}({conf_v:.2f})")

    ax.imshow(img)
    ax.set_facecolor('#1a1a2e')
    ax.axis('off')
    title = ', '.join(detections) if detections else '無偵測'
    ax.set_title(title, color='#eee', fontsize=9, pad=4)

plt.tight_layout()
plt.savefig(str(PROJECT_ROOT / 'inference_results.png'), dpi=120, bbox_inches='tight')
plt.show()
print('推論結果已儲存 → inference_results.png')

## 📦 Step 9 — 匯出 ONNX 模型（供 JetBot 使用）

In [ ]:
# 匯出為 ONNX 格式
# 後續在 JetBot 上執行:
#   trtexec --onnx=best.onnx --saveEngine=yolov4-tiny-416.trt --workspace=1024 --fp16

export_path = model_val.export(
    format   = 'onnx',
    imgsz    = IMGSZ,
    simplify = True,
    opset    = 11,      # JetPack 相容
    dynamic  = False,
    half     = False,   # FP32 (JetBot 上再轉 FP16)
)

print(f'\n✅ ONNX 模型已匯出: {export_path}')
print()
print('📋 JetBot 部署步驟:')
print('  1. 將 ONNX 檔案 scp 到 JetBot:')
print(f'     scp {export_path} jetbot@<IP>:~/yolo/')
print()
print('  2. 在 JetBot 上轉換為 TensorRT:')
print('     trtexec --onnx=best.onnx \\')
print('             --saveEngine=yolov4-tiny-416.trt \\')
print('             --workspace=1024 --fp16')
print()
print('  3. 在 ipynb 中使用:')
print('     trt_yolo = TRT_YOLO("yolov4-tiny-416", (416, 416), 4)')

## 🔄 Step 10 — 從 Checkpoint 繼續訓練（可選）

In [ ]:
# 若訓練中斷，可用此 cell 從上次的 checkpoint 繼續
# 取消下方 # 的註解並執行

# last_ckpt = PROJECT_ROOT / 'runs' / 'sign_detection' / 'weights' / 'last.pt'
# model_resume = YOLO(str(last_ckpt))
# results_resume = model_resume.train(
#     resume = True,
#     data   = str(FIXED_YAML),
#     epochs = 150,   # 增加總 epoch 數
# )
print('此 Cell 用於從 checkpoint 繼續訓練，請解除上方的註解後執行。')